# Lecture 2: Stationary Processes
- 2.1 Fundamental Properties
- 2.2 Linear Processes
- 2.3 Introduction to ARMA Processes
- 2.4 Properties of the Sample Mean and ACF
- 2.5 Prediction of Stationary Time Series (Durbin-Levinson, Innovation Algorithms)
- 2.6 The Wold Decomposition


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from scipy.linalg import toeplitz
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
print("Ready!")


## 2.1 Fundamental Properties

A process $\{X_t, t \in \mathbb{Z}\}$ is **stationary** (weakly stationary) if:
1. $E[X_t^2] < \infty$
2. $E[X_t] = \mu$ (constant for all $t$)
3. $\gamma_X(t+h, t) = \text{Cov}(X_{t+h}, X_t) = \gamma_X(h)$ (depends only on $h$)

**Autocovariance Function (ACVF):** $\gamma(h)$

**Autocorrelation Function (ACF):** $\rho(h) = \gamma(h)/\gamma(0)$

Key properties: $|\rho(h)| \leq 1$, $\gamma(-h) = \gamma(h)$ (symmetry)


In [ ]:
# ── ACVF and ACF: MA(1) example ──
# MA(1): X_t = Z_t + theta Z_{t-1}
# gamma(0) = (1+theta^2)sigma^2, gamma(1) = theta*sigma^2, gamma(h)=0 for |h|>1

sigma2 = 1.0
thetas = [0.5, -0.5, 0.9, -0.9]
lags = np.arange(0, 6)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'crimson', 'green', 'orange']

for theta, color in zip(thetas, colors):
    acvf = np.zeros(len(lags))
    acvf[0] = (1 + theta**2) * sigma2
    acvf[1] = theta * sigma2
    acf_vals = acvf / acvf[0]
    
    axes[0].stem(lags, acvf, linefmt=color, markerfmt=f'o', 
                 basefmt='k-', label=f'theta={theta}')
    axes[1].stem(lags, acf_vals, linefmt=color, markerfmt=f'o',
                 basefmt='k-', label=f'theta={theta}')

for ax, title in zip(axes, ['ACVF gamma(h)', 'ACF rho(h)']):
    ax.set_title(f'MA(1): {title}', fontsize=13)
    ax.set_xlabel('Lag h')
    ax.legend()
    ax.axhline(0, color='k', lw=0.5)

plt.tight_layout(); plt.show()
print("MA(1): rho(0)=1, rho(1)=theta/(1+theta^2), rho(h)=0 for |h|>1")
for theta in thetas:
    print(f"  theta={theta:5.1f} => rho(1) = {theta/(1+theta**2):.4f}")


## 2.2 Linear Processes

A **linear process** is defined as:
$$X_t = \sum_{j=-\infty}^{\infty} \psi_j Z_{t-j}, \quad \{Z_t\} \sim WN(0, \sigma^2)$$

The process is stationary provided $\sum |\psi_j| < \infty$.

ACVF: $\gamma(h) = \sigma^2 \sum_{j=-\infty}^{\infty} \psi_{j+h} \psi_j$

**Causality:** The process is causal if $\psi_j = 0$ for $j < 0$ (independent of future noise).


In [ ]:
# ── Linear Process: MA(infinity) approximation ──
np.random.seed(42)
n = 500
Z = np.random.normal(0, 1, n + 100)

# Different psi weights
psi_configs = {
    'MA(3)': [1, 0.5, 0.25, 0.125],
    'Exponential Decay': [0.8**j for j in range(20)],
    'Seasonal': [1 if j % 12 == 0 else 0 for j in range(25)],
}

fig, axes = plt.subplots(len(psi_configs), 2, figsize=(14, 10))

for idx, (name, psi) in enumerate(psi_configs.items()):
    psi = np.array(psi)
    q = len(psi) - 1
    
    # Generate series
    X = np.array([sum(psi[j] * Z[t + 100 - j] for j in range(len(psi))) for t in range(n)])
    
    axes[idx, 0].plot(X[:150], lw=1, color='steelblue')
    axes[idx, 0].set_title(f'{name} — Time Series (first 150 obs)')
    axes[idx, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    # Theoretical ACVF
    max_lag = 30
    gamma = np.array([sum(psi[j] * psi[j+h] for j in range(len(psi)-h)) 
                      for h in range(min(max_lag, len(psi)))])
    gamma = np.concatenate([gamma, np.zeros(max_lag - len(gamma))])
    
    axes[idx, 1].stem(range(max_lag), gamma/gamma[0], 
                       linefmt='steelblue', markerfmt='o', basefmt='k-')
    axes[idx, 1].set_title(f'{name} — Theoretical ACF')
    axes[idx, 1].axhline(0, color='k', lw=0.5)

plt.tight_layout(); plt.show()


## 2.3 Introduction to ARMA Processes

**ARMA(p, q) Model:**
$$\phi(B)X_t = \theta(B)Z_t$$

$$X_t - \phi_1 X_{t-1} - \cdots - \phi_p X_{t-p} = Z_t + \theta_1 Z_{t-1} + \cdots + \theta_q Z_{t-q}$$

where $B$ is the backshift operator: $BX_t = X_{t-1}$

- **Causality:** $\phi(z) \neq 0$ for $|z| \leq 1$
- **Invertibility:** $\theta(z) \neq 0$ for $|z| \leq 1$


In [ ]:
# ── ARMA Simulation and Characterisation ──
from statsmodels.tsa.arima_process import arma_generate_sample

np.random.seed(1)
n = 300

models = {
    'AR(1) phi=0.8':           {'ar': [1, -0.8],    'ma': [1]},
    'AR(2) phi1=0.5, phi2=0.3': {'ar': [1,-0.5,-0.3], 'ma': [1]},
    'MA(2) theta1=0.7, theta2=-0.4': {'ar': [1],    'ma': [1, 0.7, -0.4]},
    'ARMA(1,1) phi=0.6, theta=0.4': {'ar': [1,-0.6], 'ma': [1, 0.4]},
}

fig, axes = plt.subplots(len(models), 3, figsize=(16, 12))

for i, (name, params) in enumerate(models.items()):
    X = arma_generate_sample(params['ar'], params['ma'], n, scale=1)
    
    axes[i, 0].plot(X, lw=0.8, color='steelblue')
    axes[i, 0].set_title(f'{name}')
    axes[i, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    plot_acf(X, lags=20, ax=axes[i, 1], color='steelblue', title='ACF')
    plot_pacf(X, lags=20, ax=axes[i, 2], color='darkorange', title='PACF')

plt.tight_layout(); plt.show()

print("Model Identification Guide:")
print("  ACF cuts off, PACF tails off  => AR(p) model")
print("  ACF tails off, PACF cuts off  => MA(q) model")
print("  Both tail off                  => ARMA(p,q) model")


## 2.5 Prediction of Stationary Time Series

### 2.5.3 Durbin-Levinson Algorithm

Computes the PACF $\{\phi_{nn}\}$ recursively:

$$\phi_{n+1,n+1} = \frac{\rho(n+1) - \sum_{j=1}^{n}\phi_{nj}\rho(n+1-j)}{1 - \sum_{j=1}^{n}\phi_{nj}\rho(j)}$$

This yields the best linear predictor for the $(n+1)$-step forecast:
$$\hat{X}_{n+1} = \phi_{n1}X_n + \phi_{n2}X_{n-1} + \cdots + \phi_{nn}X_1$$


In [ ]:
# ── Durbin-Levinson Algorithm (manual implementation) ──

def durbin_levinson(rho, n_steps):
    '''
    rho: ACF values [rho(0), rho(1), rho(2), ...]
    Returns: PACF values phi_{nn}
    '''
    phi = {}   # phi[n][j] = phi_{n,j}
    v = {}     # prediction error variances
    pacf_vals = []
    
    phi[1] = {1: rho[1]}
    v[1] = 1 - rho[1]**2
    pacf_vals.append(rho[1])
    
    for n in range(1, n_steps):
        num = rho[n+1] - sum(phi[n].get(j,0) * rho[n+1-j] for j in range(1, n+1))
        denom = v[n]
        phi_new = num / denom
        pacf_vals.append(phi_new)
        
        phi[n+1] = {}
        phi[n+1][n+1] = phi_new
        for j in range(1, n+1):
            phi[n+1][j] = phi[n].get(j,0) - phi_new * phi[n].get(n+1-j, 0)
        
        v[n+1] = v[n] * (1 - phi_new**2)
    
    return np.array(pacf_vals), v

# Theoretical ACF of an AR(2) process
phi1, phi2, sigma2 = 0.7, -0.3, 1.0
# Theoretical ACF from Yule-Walker:
rho1 = phi1 / (1 - phi2)
rho2 = phi2 + phi1 * rho1
rho_vals = [1.0, rho1, rho2]
for h in range(3, 25):
    rho_vals.append(phi1 * rho_vals[-1] + phi2 * rho_vals[-2])

pacf_dl, variances = durbin_levinson(rho_vals, 15)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].stem(range(len(rho_vals)), rho_vals, linefmt='steelblue', 
             markerfmt='o', basefmt='k-')
axes[0].set_title('Theoretical ACF — AR(2) phi1=0.7, phi2=-0.3')
axes[0].set_xlabel('Lag h')

axes[1].stem(range(1, len(pacf_dl)+1), pacf_dl, linefmt='darkorange', 
             markerfmt='o', basefmt='k-')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title('Durbin-Levinson PACF')
axes[1].set_xlabel('Lag n')
axes[1].axhline(phi1, color='red', ls='--', alpha=0.5, label=f'phi1={phi1}')
axes[1].axhline(phi2, color='blue', ls='--', alpha=0.5, label=f'phi2={phi2}')
axes[1].legend()

axes[2].plot(list(variances.values()), 'g-o', markersize=5)
axes[2].set_title('Prediction Error Variance $v_n$')
axes[2].set_xlabel('n')
axes[2].axhline(sigma2*(1-phi1**2-phi2**2-phi1*phi2*rho1), color='red', 
                ls='--', label='Theoretical minimum')
axes[2].legend()

plt.tight_layout(); plt.show()
print(f"PACF phi11 = {pacf_dl[0]:.4f} (Expected: {rho1:.4f})")
print(f"PACF phi22 = {pacf_dl[1]:.4f} (Expected: {phi2:.4f})")
print(f"PACF phi33 = {pacf_dl[2]:.6f} (Expected: approximately 0)")


## 2.6 The Wold Decomposition

**Theorem (Wold, 1938):** Every zero-mean stationary process $\{X_t\}$ can be written as:
$$X_t = \sum_{j=0}^{\infty} \psi_j Z_{t-j} + V_t$$

where:
- $\{Z_t\} \sim WN(0, \sigma^2)$: the innovation process
- $\{V_t\}$: deterministic process (uncorrelated with $Z_t$)
- $\sum \psi_j^2 < \infty$, $\psi_0 = 1$

Let us illustrate this theorem with an AR(1) example:


In [ ]:
# ── Wold Decomposition: AR(1) → MA(infinity) ──

phi = 0.7
max_terms = [3, 10, 50, 200]

np.random.seed(42)
Z = np.random.normal(0, 1, 1000)
n_show = 100

fig, axes = plt.subplots(len(max_terms)+1, 1, figsize=(13, 12), sharex=True)

# True AR(1)
X_true = np.zeros(1000)
for t in range(1, 1000):
    X_true[t] = phi * X_true[t-1] + Z[t]

axes[0].plot(X_true[900:], 'k', lw=1.5, label='True AR(1)')
axes[0].set_title('AR(1): $X_t = 0.7 X_{t-1} + Z_t$')
axes[0].legend(); axes[0].axhline(0, color='gray', ls='--', lw=0.5)

# Wold MA(infinity) approximation
for idx, q in enumerate(max_terms):
    psi = np.array([phi**j for j in range(q)])
    X_approx = np.array([sum(psi[j] * Z[t-j] for j in range(min(q, t+1))) 
                         for t in range(900, 1000)])
    axes[idx+1].plot(X_approx, color='steelblue', lw=1, alpha=0.8, label=f'MA(inf) q={q}')
    axes[idx+1].plot(X_true[900:], 'k--', lw=0.8, alpha=0.4, label='True')
    axes[idx+1].set_title(f'Wold Approximation: q={q} terms, psi_j = 0.7^j')
    axes[idx+1].legend(fontsize=9)
    axes[idx+1].axhline(0, color='gray', ls='--', lw=0.5)
    
    mse = np.mean((X_approx - X_true[900:])**2)
    print(f"q={q:4d} terms: MSE = {mse:.6f}")

axes[-1].set_xlabel('Time')
plt.tight_layout(); plt.show()
print("\nAs q increases, the MA(inf) approximation converges to AR(1)!")


## Summary

| Concept | Formula | Importance |
|---------|---------|------------|
| **Stationarity** | $\gamma(t+h,t)=\gamma(h)$ | Fundamental assumption |
| **ACVF** | $\gamma(h) = \text{Cov}(X_{t+h}, X_t)$ | Covariance structure |
| **ACF** | $\rho(h) = \gamma(h)/\gamma(0)$ | Correlation structure |
| **PACF** | $\phi_{nn}$ | Direct effect |
| **Durbin-Levinson** | Recursive computation of $\phi_{nn}$ | Efficient prediction |
| **Wold Decomposition** | $X_t = \sum \psi_j Z_{t-j} + V_t$ | Foundation for ARMA modelling |

